<a href="https://colab.research.google.com/github/tuckerlucy1/HLS-Data-Resources/blob/main/Quality_trial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import json

drive.mount('/content/drive')
# Create a folder for your quantum data
!mkdir -p "/content/drive/My Drive/Quantum_Experiments"


In [ ]:

# --- STEP 1: INSTALLATION ---
# Run this once, then RESTART the Colab Session
!pip install -U qiskit qiskit-ibm-runtime matplotlib pylatexenc

# --- STEP 2: THE ENGINE ---
import numpy as np
import matplotlib.pyplot as plt
from google.colab import userdata # For secure token handling

from qiskit.circuit.library import IQP, XGate
from qiskit.quantum_info import random_hermitian, SparsePauliOp, Statevector
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator, EstimatorOptions, Session
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes.scheduling import ALAPScheduleAnalysis, PadDynamicalDecoupling

# 1. Setup Service (Uses Colab Secrets)
token = userdata.get('IBM_TOKEN')
service = QiskitRuntimeService(channel="ibm_quantum", token=token)
backend = service.least_busy(operational=True, simulator=False)
print(f"Using Backend: {backend.name}")

# 2. Hardware Analysis: Find the cleanest 1D string of qubits
target = backend.target
gate_name = 'cx' if 'cx' in target.operation_names else 'ecr'
gate_errors = []
for q_tuple in target.instruction_properties(gate_name):
    error = target.instruction_properties(gate_name)[q_tuple].error
    gate_errors.append((q_tuple, error))

# Sort to find a chain of low-error qubits (simplified logic)
gate_errors.sort(key=lambda x: x[1])
golden_chain = list(gate_errors[0][0]) # Starting with the best pair

# 3. Scaling Loop Configuration
qubit_range = range(3, 7)
results = {"qubits": [], "hw_evs": [], "theo_evs": [], "stds": []}

options = EstimatorOptions()
options.resilience_level = 2 # Readout + Bias mitigation
options.default_shots = 8192

# 4. Run the Experiment
with Session(backend=backend) as session:
    estimator = Estimator(mode=session, options=options)

    for n in qubit_range:
        print(f"Running n={n}...")

        # A. Create IQP and Theory
        mat = np.real(random_hermitian(n, seed=42))
        qc = IQP(mat)
        obs = SparsePauliOp("Z" * n)
        theo_ev = Statevector.from_instruction(qc).expectation_value(obs).real

        # B. Hardware-Aware Pass Manager
        pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
        durations = backend.target.durations()

        # ADD DYNAMICAL DECOUPLING (The Shield)
        pm.add_pass(ALAPScheduleAnalysis(durations))
        pm.add_pass(PadDynamicalDecoupling(durations, [XGate(), XGate()]))

        isa_qc = pm.run(qc)
        isa_obs = obs.apply_layout(isa_qc.layout)

        # C. Execute
        job = estimator.run([(isa_qc, isa_obs)])
        pub_res = job.result()[0]

        results["qubits"].append(n)
        results["hw_evs"].append(pub_res.data.evs)
        results["theo_evs"].append(theo_ev)
        results["stds"].append(pub_res.data.stds)

# --- STEP 3: PLOTTING ---
plt.errorbar(results["qubits"], results["hw_evs"], yerr=results["stds"], fmt='-o', label='Hardware')
plt.plot(results["qubits"], results["theo_evs"], 'r--', label='Ideal')
plt.title(f"IQP Scaling on {backend.name}")
plt.xlabel("Qubits")
plt.ylabel("Expectation Value")
plt.legend()
plt.show()




In [ ]:
file_path = f"/content/drive/My Drive/Quantum_Experiments/IQP_Scaling_{backend.name}.json"

with open(file_path, 'w') as f:
    # We convert numpy arrays to lists so JSON can handle them
    json_ready_results = {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in results.items()}
    json.dump(json_ready_results, f)

print(f"✅ Data safely tucked away in your Drive at: {file_path}")
